In [ ]:
import xwrf
import glob
import pickle

import xarray as xr
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import numpy as np
from wrf import latlon_coords, ll_to_xy, xy_to_ll
from netCDF4 import Dataset

import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error

from datetime import datetime

Calculate the performance metrics for the four experiments - Tables 4, S1-S6

The metrics include mean bias error, mean absolute error, and Pearson correlation.

In [ ]:
#load extracted WRF data at stations
with open("/g/data/gb02/mf9078/WRF_Results-10-20Jan_xy_wtime_DCCEEW_mean.pkl", "rb") as f:
    wrf_stations = pickle.load(f)

In [ ]:
#load obs data in the desired format

# Read the CSV file
df = pd.read_csv('/g/data/gb02/mf9078/T-DCCEEW.csv')

# Combine Date and Time columns into a single datetime column
#df['Time'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M')
df['Time'] = pd.to_datetime(df['Date'] + ' ' + df['Time'].astype(str), dayfirst=True)

# Drop the original Date and Time columns (optional)
df = df.drop(['Date'], axis=1)

# Reorder columns to put Time first
cols = ['Time'] + [col for col in df.columns if col != 'Time']
df = df[cols]
# Drop the last 24 hours (info on 20 Jan not needed)
df = df.iloc[:-24]

In [ ]:
#calculate the metrics at each station + the total spatial average
# Station codes
stations = ['RK', 'RE', 'LL', 'CA', 'ED', 'SS', 'PT', 'CN']

# Scenarios
scenarios = ['Default(Bulk+NoUCM)', 'LCZ', 'WSF-MB', 'Geoscape']

# Delete time key for easier manipulation
del wrf_stations['time']

# Initialize dictionaries to store results
results = {
    'Station': [],
    'Scenario': [],
    'Mean Bias': [],
    'MAE': [],
    'Correlation': []
}

# Calculate metrics for each scenario and station
for scenario in scenarios:
    for station in stations:
        # Get observed and modelled data
        obs_temps = df[station].values
        wrf_temps = np.array(wrf_stations[scenario][station])
        
        # Remove any NaN values
        valid_mask = ~(np.isnan(obs_temps) | np.isnan(wrf_temps))
        obs_temps_valid = obs_temps[valid_mask]
        wrf_temps_valid = wrf_temps[valid_mask]
        
        # Calculate metrics
        bias = np.mean(wrf_temps_valid - obs_temps_valid)
        mae = mean_absolute_error(obs_temps_valid, wrf_temps_valid)
        correlation, _ = pearsonr(obs_temps_valid, wrf_temps_valid)
        
        # Store results
        results['Station'].append(station)
        results['Scenario'].append(scenario)
        results['Mean Bias'].append(bias)
        results['MAE'].append(mae)
        results['Correlation'].append(correlation)

# Create DataFrame with results
metrics_df = pd.DataFrame(results)

# Save results
metrics_df.to_csv("/g/data/gb02/mf9078/plots/WRF_vs_Station_Comparison_DCCEEW_mean.csv", index=False)
print("Analysis complete. Results saved to 'WRF_vs_Station_Comparison_DCCEEW_mean.csv'.")

#average metrics over all stations for each scenario
df_avg_metrics = metrics_df.groupby('Scenario')[['Mean Bias', 'MAE', 'Correlation']].mean()


# Save the results
df_avg_metrics.to_csv("/g/data/gb02/mf9078/plots/WRF_Average_Metrics_DCCEEW_mean.csv", index=False)
print("Average metrics per experiment saved to 'WRF_Average_Metrics_DCCEEW_mean.csv'.")

Analysis complete. Results saved to 'WRF_vs_Station_Comparison_DCCEEW_mean.csv'.
Average metrics per experiment saved to 'WRF_Average_Metrics_DCCEEW_mean.csv'.


In [14]:
df_avg_metrics

,Mean Bias,MAE,Correlation
Scenario,,,
Default(Bulk+NoUCM),1.339117,1.745724,0.929423
Geoscape,0.828474,1.493602,0.928435
LCZ,1.113039,1.622911,0.925489
WSF-MB,0.893528,1.547261,0.924278
